# 01 · Data Loading & Exploratory Data Analysis

Load the S&P 500 dataset, filter to 2010–2024, create the binary direction target, and produce core EDA charts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': config.FIG_DPI})
print('Libraries loaded ✓')

## 1. Load Data

We first try to read the Kaggle CSV (`data/sp500_index.csv`). If the file is absent we fall back to **yfinance**.

In [ ]:
def _generate_synthetic_sp500(start: str, end: str, seed: int = 42) -> pd.DataFrame:
    """
    Generate synthetic S&P 500-like OHLCV data using geometric Brownian motion.
    Used when neither the Kaggle CSV nor yfinance is available (e.g., offline env).
    Parameters approximate historical S&P 500 behaviour (2010-2024).
    """
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, end=end)
    n = len(dates)

    # GBM parameters (~historical S&P 500 annualised stats)
    mu_daily    = 0.00045    # ~11.3% annualised return
    sigma_daily = 0.011      # ~17.5% annualised volatility
    s0          = 1257.60    # approx Jan-2010 S&P 500 level

    # Daily close-to-close log returns
    log_returns = rng.normal(mu_daily, sigma_daily, size=n)
    close = s0 * np.exp(np.cumsum(log_returns))

    # Intra-day: generate realistic O/H/L from close
    intraday_vol = sigma_daily * 0.6
    open_  = close * np.exp(rng.normal(0, intraday_vol * 0.5, size=n))
    high   = np.maximum(open_, close) * np.exp(np.abs(rng.normal(0, intraday_vol, size=n)))
    low    = np.minimum(open_, close) * np.exp(-np.abs(rng.normal(0, intraday_vol, size=n)))
    volume = (rng.integers(2_000_000_000, 5_000_000_000, size=n)).astype(float)

    df_syn = pd.DataFrame({
        'Open': open_, 'High': high, 'Low': low,
        'Close': close, 'Volume': volume,
    }, index=dates)
    df_syn.index.name = 'Date'
    print(f'Generated synthetic S&P 500 data: {n:,} trading days')
    return df_syn


def load_sp500(csv_path: str, ticker: str = '^GSPC',
               start: str = '2010-01-01', end: str = '2024-12-31') -> pd.DataFrame:
    """Load S&P 500 OHLCV data from a CSV or yfinance, filtered to [start, end].
    Falls back to synthetic data when both sources are unavailable."""
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, parse_dates=True)
        # Normalise column names
        df.columns = [c.strip().title() for c in df.columns]
        # Identify the date column
        date_cols = [c for c in df.columns if 'date' in c.lower()]
        if date_cols:
            df = df.rename(columns={date_cols[0]: 'Date'})
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date').sort_index()
        print(f'Loaded CSV: {csv_path}')
    else:
        try:
            import yfinance as yf
            raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
            # Flatten MultiIndex columns if present
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)
            raw.index.name = 'Date'
            if len(raw) > 0:
                df = raw
                print(f'Downloaded {ticker} from yfinance')
            else:
                raise ValueError('yfinance returned empty data')
        except Exception as exc:
            print(f'yfinance unavailable ({exc}); using synthetic data.')
            df = _generate_synthetic_sp500(start, end)

    # Ensure required columns exist
    required = {'Open', 'High', 'Low', 'Close', 'Volume'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {missing}')

    # Filter date range & clean
    df = df.loc[start:end].copy()
    df = df.dropna(subset=['Open', 'Close'])
    # Coerce all OHLCV to numeric (some CSVs may load as strings)
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['Open', 'Close'])
    return df


df = load_sp500(
    csv_path=config.KAGGLE_CSV,
    ticker=config.YFINANCE_TICKER,
    start=config.START_DATE,
    end=config.END_DATE,
)
print(f'Shape: {df.shape}')
df.tail(3)

## 2. Create Binary Target

`Direction = 1` if **Close > Open** (bullish day), else `0` (bearish / flat).

In [ ]:
def create_target(df: pd.DataFrame) -> pd.DataFrame:
    """Add Direction column and log-return column to df."""
    df = df.copy()
    df['Direction'] = (df['Close'] > df['Open']).astype(int)
    df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna(subset=['LogReturn'])
    return df


df = create_target(df)
print('Direction distribution:')
print(df['Direction'].value_counts(normalize=True).rename({1: 'Up', 0: 'Down'}).round(3))
df.head(3)

## 3. Price Chart

In [ ]:
fig, ax = plt.subplots(figsize=config.FIG_SIZE)
ax.plot(df.index, df['Close'], linewidth=0.8, color='steelblue')
ax.set_title('S&P 500 Closing Price (2010–2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 4. Daily Log Returns

In [ ]:
fig, ax = plt.subplots(figsize=config.FIG_SIZE)
ax.plot(df.index, df['LogReturn'], linewidth=0.5, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('S&P 500 Daily Log Returns (2010–2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print(df['LogReturn'].describe().round(5))

## 5. Rolling 20-Day Volatility

In [ ]:
df['RollingVol'] = df['LogReturn'].rolling(window=config.VOL_WINDOW).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=config.FIG_SIZE)
ax.plot(df.index, df['RollingVol'], linewidth=0.8, color='tomato')
ax.set_title(f'Rolling {config.VOL_WINDOW}-Day Annualised Volatility')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (annualised)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Add MA, volatility, RSI, and Bollinger Band features."""
    d = df.copy()
    d['MA_short'] = d['Close'].rolling(config.MA_SHORT).mean()
    d['MA_long']  = d['Close'].rolling(config.MA_LONG).mean()
    d['MA_cross'] = d['MA_short'] - d['MA_long']
    d['Volatility'] = d['LogReturn'].rolling(config.VOL_WINDOW).std()

    # RSI
    delta = d['Close'].diff()
    gain  = delta.clip(lower=0).rolling(config.RSI_WINDOW).mean()
    loss  = (-delta.clip(upper=0)).rolling(config.RSI_WINDOW).mean()
    rs    = gain / loss.replace(0, np.nan)
    d['RSI'] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    bb_mid  = d['Close'].rolling(config.BB_WINDOW).mean()
    bb_std  = d['Close'].rolling(config.BB_WINDOW).std()
    d['BB_upper'] = bb_mid + config.BB_STD * bb_std
    d['BB_lower'] = bb_mid - config.BB_STD * bb_std
    d['BB_width'] = (d['BB_upper'] - d['BB_lower']) / bb_mid
    d['BB_pct']   = (d['Close'] - d['BB_lower']) / (d['BB_upper'] - d['BB_lower'])

    return d.dropna()


df_feat = add_technical_indicators(df)

feat_cols = ['LogReturn', 'MA_cross', 'Volatility', 'RSI',
             'BB_width', 'BB_pct', 'Direction']
corr = df_feat[feat_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap – Features & Target')
plt.tight_layout()
plt.show()

## 7. Target Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Direction'].value_counts().sort_index().rename({1: 'Up (1)', 0: 'Down (0)'})
ax.bar(counts.index, counts.values, color=['tomato', 'steelblue'], edgecolor='white')
ax.set_title('Up vs Down Days (2010-2024)')
ax.set_ylabel('Count')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=0)
for i, v in enumerate(counts.values):
    ax.annotate(f'{v:,.0f}', (i, v), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 8. Save Processed Dataset

In [ ]:
out_path = os.path.join(config.DATA_DIR, 'sp500_processed.csv')
df_feat.to_csv(out_path)
print(f'Saved processed data to {out_path}  ({len(df_feat):,} rows)')